# Блендинг сабмитов (Data Fusion) — шаг 4 пайплайна

**После** `solution1` → `solution2` → `pipeline1st.ipynb` положите в эту папку три CSV (или выполните из корня репо: `python scripts/collect_blend_inputs.py`).

Несколько CSV с колонками **`event_id`** и **`predict`** последовательно объединяются в итоговый файл.

**Схема:** логиты → **z-score** → взвешенное среднее в z-пространстве → **sigmoid** (стабильнее простого среднего по `predict`).

| Шаг | Файл A | Файл B | Результат | Вес A (`w`) | `eps` |
|-----|--------|--------|-----------|-------------|-------|
| 1 | `submission_DL_PUBLIC.csv` | `submission_ICEQ_PUBLIC.csv` | `BLEND1.csv` | 0.55 | 1e-7 |
| 2 | `BLEND1.csv` | `submission_MINE.csv` | `BLEND2.csv` | 0.55 | 1e-7 |
| 3 | `BLEND2.csv` | `BLEND1.csv` | `BLEND4.csv` | 0.5485 | 1e-8 |
| 4 | `BLEND1.csv` | `BLEND4.csv` | `sub_totalblend.csv` | 0.541415926 | 1e-9 |

Тот же код без Jupyter: `python blend_submissions.py` из каталога `AGI/`.

## Код: функции и запуск конвейера

Ниже — одна ячейка: определения и цикл по шагам. При необходимости можно закомментировать часть строк в `BLEND_STEPS` и гонять этапы по отдельности.

In [1]:
import numpy as np
import pandas as pd


def logit(p, eps):
    p = np.clip(p, eps, 1 - eps)
    return np.log(p / (1 - p))


def zscore(x):
    x = np.asarray(x, dtype=float)
    return (x - x.mean()) / (x.std() + 1e-12)


def sigmoid(x):
    return 1 / (1 + np.exp(-x))


def blend_two_csvs(path_a, path_b, out_path, weight_a, eps=1e-7):
    """Смесь двух сабмитов: weight_a — доля первой модели в z-пространстве логитов."""
    a = pd.read_csv(path_a).rename(columns={"predict": "pred_a"})
    b = pd.read_csv(path_b).rename(columns={"predict": "pred_b"})
    df = a.merge(b, on="event_id", how="inner")
    za = zscore(logit(df["pred_a"].values, eps))
    zb = zscore(logit(df["pred_b"].values, eps))
    z = weight_a * za + (1 - weight_a) * zb
    out = df[["event_id"]].copy()
    out["predict"] = sigmoid(z)
    out.to_csv(out_path, index=False)
    print(f"OK → {out_path}  ({len(out):,} строк)")
    return out


BLEND_STEPS = [
    # path_a, path_b, out_path, weight_a, eps
    ("submission_DL_PUBLIC.csv", "submission_ICEQ_PUBLIC.csv", "BLEND1.csv", 0.55, 1e-7),
    ("BLEND1.csv", "submission_MINE.csv", "BLEND2.csv", 0.55, 1e-7),
    ("BLEND2.csv", "BLEND1.csv", "BLEND4.csv", 0.5485, 1e-8),
    ("BLEND1.csv", "BLEND4.csv", "sub_totalblend.csv", 0.541415926, 1e-9),
]

for pa, pb, outp, w, ep in BLEND_STEPS:
    blend_two_csvs(pa, pb, outp, w, ep)

OK → BLEND1.csv  (633,683 строк)
OK → BLEND2.csv  (633,683 строк)
OK → BLEND4.csv  (633,683 строк)
OK → sub_totalblend.csv  (633,683 строк)
